# How does a pair react to a central bank decision? An event study

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Interest-Rate-Decision/notebooks/blob/main/01-event-study.ipynb)

Each bank's decisions on the pair [InterestRateDecision.com](https://interestratedecision.com/?via=notebooks) measures it on (the Fed and the ECB on EUR/USD,
the BoE on GBP/USD, the BoJ on USD/JPY...): the path of price from **three hours before** each announcement to
**five hours after**, in pips from the last close before it, averaged by what the bank did: **hike, hold or cut**.

Paths are turned so that **up means the bank's currency rose**: for the Fed on EUR/USD, a falling EUR/USD is a rising dollar.
Pips only compare within one pair: a pip on EUR/NOK is a tenth of the size of one on EUR/USD.

### Candles: download your own

Dukascopy's candles are not ours to share, so this repository ships none (only a synthetic sample). Two ways to get them:

1. **The `dukascopy-node` CLI** (needs Node.js): `python scripts/download_candles.py --banks fed ecb --since 2020-01-01` runs
   `npx dukascopy-node -i eurusd -from ... -to ... -t m1 -f csv` for the hours around each decision and writes `candles/EURUSD/`.
   Add `--dry-run` to see the commands first.
2. **By hand** from Dukascopy's historical data export (one-minute, bid, GMT): save each CSV in `candles/<PAIR>/`.

`ird.read_candle_csv` reads both formats. Then set `SAMPLE = False` below.


In [ ]:
# On Colab or Kaggle, fetch the repository first: the helper module and the synthetic sample.
import os, subprocess, sys
if not os.path.exists("ird_events.py"):
    if not os.path.exists("notebooks-repo"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Interest-Rate-Decision/notebooks.git", "notebooks-repo"], check=True)
    os.chdir("notebooks-repo")
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ird_events as ird

In [ ]:
SAMPLE = True                    # True: synthetic candles from data/sample, offline. False: your own downloads.
CANDLES_DIR = "candles"          # scripts/download_candles.py writes candles/<PAIR>/*.csv
BANKS = ["fed", "ecb", "boe"]    # any of: fed ecb boe boj boc rba rbnz snb norges riksbank
SINCE = "2015-01-01"             # IRD's decisions go back to 2008

ird.banner(SAMPLE)
decisions = ird.load_decisions(sample=SAMPLE)
decisions = decisions[decisions.bank.isin(BANKS) & (decisions.time_utc >= pd.Timestamp(SINCE, tz="UTC"))]
candles = {
    pair: ird.load_sample_candles(pair) if SAMPLE else ird.load_candles(pair, CANDLES_DIR)
    for pair in decisions.pair.dropna().unique()
}
print(len(decisions), "decisions;", {pair: len(c) for pair, c in candles.items()}, "one-minute candles")

## The path of each decision

One row per decision, one column per minute from −180 to +299.

In [ ]:
paths, rows = [], []
for d in decisions.itertuples():
    info = ird.BANKS[d.bank]
    p = ird.path_in_pips(candles[d.pair], d.time_utc, d.pair, info["side"])
    if p is None:
        continue  # no candles for this one: not downloaded, or a gap
    paths.append(p.rename((d.bank, d.date)))
    rows.append({"bank": d.bank, "date": d.date, "move": d.move, "change_bp": d.change_bp})
paths = pd.DataFrame(paths)
meta = pd.DataFrame(rows)
print(len(paths), "of", len(decisions), "decisions have candles")
meta.groupby(["bank", "move"]).size().unstack(fill_value=0)

## The average path, by bank and by move

In [ ]:
moves = {"hike": "tab:red", "hold": "tab:gray", "cut": "tab:blue"}
banks = list(meta.bank.unique())
fig, axes = plt.subplots(len(banks), 1, figsize=(10, 3.4 * len(banks)), squeeze=False)
for ax, bank in zip(axes[:, 0], banks):
    for move, colour in moves.items():
        mask = (meta.bank == bank) & (meta.move == move)
        if mask.sum() == 0:
            continue
        ax.plot(paths.columns, paths[mask.to_numpy()].mean(), color=colour, label=f"{move} (n={mask.sum()})")
    ax.axvline(0, color="black", lw=0.8)
    ax.axhline(0, color="black", lw=0.5, alpha=0.4)
    ax.set_title(f"{ird.BANKS[bank]['name']} on {ird.BANKS[bank]['pair']}: average path, pips (up = {ird.BANKS[bank]['currency']} stronger)")
    ax.set_xlabel("minutes from the announcement")
    ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

## At set times

Mean and median in pips at a few minutes, and how many decisions each figure counts. Medians are steadier: one big surprise moves a mean.

In [ ]:
at = [-60, -15, 1, 5, 15, 60, 240]
table = paths[at].copy()
table.insert(0, "move", meta.move.to_numpy())
table.insert(0, "bank", meta.bank.to_numpy())
grouped = table.groupby(["bank", "move"])
summary = grouped[at].agg(["mean", "median"]).round(1)
summary[("n", "")] = grouped.size()
summary

## How spread out it is

The average hides the spread: here every path of one bank, with the median on top.

In [ ]:
bank = banks[0]
mask = (meta.bank == bank).to_numpy()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(paths.columns, paths[mask].T, color="tab:gray", alpha=0.25, lw=0.8)
ax.plot(paths.columns, paths[mask].median(), color="black", lw=2, label="median")
ax.axvline(0, color="black", lw=0.8)
ax.set_title(f"Every {ird.BANKS[bank]['name']} decision, pips (up = {ird.BANKS[bank]['currency']} stronger)")
ax.set_xlabel("minutes from the announcement")
ax.legend()
plt.show()

## Reading it

- The move a bank makes is usually expected: a cut that everyone expected can lift the bank's currency. What moves price is the gap between the decision and what was expected (notebook 02) and what the bank says.
- The minutes after the announcement are where the spread is widest and fills are worst. Notebook 03 looks at them the way InterestRateDecision.com does: the hour before, the first spike, the close back.
- A few decisions per bank and move is a small sample: read the `n` column before the line.

---

Decisions from [InterestRateDecision.com](https://interestratedecision.com/?via=notebooks) (free to reuse with a link). The method: [how the site measures a reaction](https://interestratedecision.com/about/?via=notebooks#method). The studies: [how often central banks surprise](https://interestratedecision.com/blog/how-often-do-central-banks-surprise/?via=notebooks), [the most volatile forex news](https://interestratedecision.com/blog/most-volatile-forex-news/?via=notebooks), [how to trade FOMC](https://interestratedecision.com/blog/how-to-trade-fomc/?via=notebooks), [can central bank decisions be predicted?](https://interestratedecision.com/blog/can-central-bank-decisions-be-predicted/?via=notebooks). Learn to test and trade the news: [Forexia's courses](https://forexiapro.com/courses?via=notebooks).

Not financial advice. Past reactions say nothing certain about the next one.

Part of the Forexia network: [InterestRateDecision.com](https://interestratedecision.com/?via=notebooks) · [Forexia Academy](https://forexiapro.com/courses?via=notebooks) · [TradingSignalBot](https://tradingsignalbot.com/?via=notebooks) · [PropSim](https://github.com/PropSim-Enterprise) · [OpenCharts](https://github.com/dylanpersonguy/OpenCharts)